In [24]:
# ============================================
# CELL 1 — MASTER DATA SOURCE
# Single source of truth — change here only
# ============================================

import pandas as pd
import numpy as np
import yfinance as yf

# --- Pull live data from yfinance ---
fc = yf.Ticker("FCH.L")

income = fc.financials / 1e6        # convert to £m
balance = fc.balance_sheet / 1e6

years = [2023, 2024, 2025]

def get_metric(df, row, year):
    col = f"{year}-12-31"
    try:
        return round(df.loc[row, col], 1)
    except:
        return np.nan

# --- Section A: Auto-fetched from yfinance ---
REVENUE      = {y: get_metric(income, 'Total Revenue', y) for y in years}
EBIT         = {y: get_metric(income, 'EBIT', y) for y in years}
EBITDA       = {y: get_metric(income, 'EBITDA', y) for y in years}
PRETAX_PROFIT= {y: get_metric(income, 'Pretax Income', y) for y in years}
TOTAL_DEBT   = {y: get_metric(balance, 'Total Debt', y) for y in years}
CASH         = {y: get_metric(balance, 'Cash And Cash Equivalents', y) for y in years}
TOTAL_ASSETS = {y: get_metric(balance, 'Total Assets', y) for y in years}
EQUITY       = {y: get_metric(balance, 'Stockholders Equity', y) for y in years}
OPEX         = {y: get_metric(income, 'Operating Expense', y) for y in years}

# --- Section B: Manual overrides (from Annual Report) ---
# Only change these when a new annual report is published
# Sources noted for each line

BANK_BORROWINGS = {      # Note 16, 2025 Annual Report
    2023: 69.5,
    2024: 101.9,
    2025: 267.3
}

LEASE_INTEREST = {       # Note 11, 2025 Annual Report
    2023: 0.0,
    2024: 0.6,
    2025: 0.6
}

ECL = {                  # Note 16, ECL provision table
    2023: np.nan,
    2024: 8.6,
    2025: 18.3
}

NPL = {                  # Note 16, Stage 3 non-performing
    2023: np.nan,
    2024: 9.4,
    2025: 20.2
}

LOAN_BOOK = {            # Note 12, SME loans total
    2023: np.nan,
    2024: 118.2,
    2025: 307.7
}

TERM_LOAN_ORIGINATIONS = {   # Company KPIs, Annual Report
    2023: 1060.0,
    2024: 1407.0,
    2025: 1638.0
}

FLEXIPAY_TRANSACTIONS = {    # Company KPIs, Annual Report
    2023: 234.0,
    2024: 492.0,
    2025: 815.0
}

# --- Section C: Market assumptions ---
# Update when SONIA rate changes materially
SONIA_RATE = 0.052           # Bank of England base, approximate avg
BORROWING_SPREAD = 0.013     # Estimated spread to 6.5% cap, Note 16

# --- Build master dataframe ---
master = pd.DataFrame({
    'Revenue':              REVENUE,
    'EBIT':                 EBIT,
    'EBITDA':               EBITDA,
    'Pretax_Profit':        PRETAX_PROFIT,
    'Total_Debt':           TOTAL_DEBT,
    'Bank_Borrowings':      BANK_BORROWINGS,
    'Lease_Interest':       LEASE_INTEREST,
    'Cash':                 CASH,
    'Total_Assets':         TOTAL_ASSETS,
    'Equity':               EQUITY,
    'Opex':                 OPEX,
    'ECL':                  ECL,
    'NPL':                  NPL,
    'Loan_Book':            LOAN_BOOK,
    'Term_Loan_Originations': TERM_LOAN_ORIGINATIONS,
    'FlexiPay_Transactions':  FLEXIPAY_TRANSACTIONS,
})
# ============================================
# REVENUE OVERRIDE — IMPORTANT
# ============================================
# yfinance reports £222.5m for 2025 — incorrect
# Reason: yfinance includes fair value movements on
# FVTPL loan book which are not operating revenue
# Company reported revenue per 2025 Annual Report: £204.3m
# This is the figure used by analysts and credit committees
# Rule: always use company reported audited figures
# Source: Funding Circle FY2025 Results, 5 March 2026

master['Revenue'] = pd.Series({
    2023: 130.1,
    2024: 160.1,
    2025: 204.3
})

print("Revenue overridden with company reported figures.")
print(master['Revenue'])

master.index.name = 'Year'

print("=" * 50)
print("MASTER DATA LOADED SUCCESSFULLY")
print("=" * 50)
print(master.to_string())
print("\nSection A (yfinance): Revenue, EBIT, EBITDA, Debt, Cash, Assets, Equity, Opex")
print("Section B (manual):   Bank Borrowings, Lease Interest, ECL, NPL, Loan Book, KPIs")
print("Section C (assumed):  SONIA Rate, Borrowing Spread")

Revenue overridden with company reported figures.
2023    130.1
2024    160.1
2025    204.3
Name: Revenue, dtype: float64
MASTER DATA LOADED SUCCESSFULLY
      Revenue  EBIT  EBITDA  Pretax_Profit  Total_Debt  Bank_Borrowings  Lease_Interest   Cash  Total_Assets  Equity   Opex   ECL   NPL  Loan_Book  Term_Loan_Originations  FlexiPay_Transactions
Year                                                                                                                                                                                             
2023    130.1  -9.9     6.8           -9.9        69.5             69.5             0.0  221.4         373.2   246.8   71.4   NaN   NaN        NaN                  1060.0                  234.0
2024    160.1   3.8    16.8            0.8       109.5            101.9             0.6  187.6         358.0   216.5  105.1   8.6   9.4      118.2                  1407.0                  492.0
2025    204.3  20.4    31.4           20.3       273.6            267.

In [25]:
# ============================================
# CELL 2 — RATIO ANALYSIS
# All inputs from master — do not hardcode
# ============================================

ratios = pd.DataFrame(index=years)

# --- Profitability ---
ratios['Revenue_Growth_%'] = (
    (master['Revenue'] - master['Revenue'].shift(1)) / 
    master['Revenue'].shift(1) * 100
).round(1)

ratios['Profit_Margin_%'] = (
    master['Pretax_Profit'] / master['Revenue'] * 100
).round(1)

ratios['ROE_%'] = (
    master['Pretax_Profit'] / master['Equity'] * 100
).round(1)

ratios['ROA_%'] = (
    master['Pretax_Profit'] / master['Total_Assets'] * 100
).round(1)

# --- Efficiency ---
ratios['Cost_to_Income_%'] = (
    master['Opex'] / master['Revenue'] * 100
).round(1)

# --- Credit Quality ---
ratios['ECL_as_%_of_Revenue'] = (
    master['ECL'] / master['Revenue'] * 100
).round(1)

ratios['NPL_Ratio_%'] = (
    master['NPL'] / master['Loan_Book'] * 100
).round(1)

ratios['ECL_Coverage_%'] = (
    master['ECL'] / master['NPL'] * 100
).round(1)

# --- Leverage ---
ratios['Debt_to_Equity'] = (
    master['Bank_Borrowings'] / master['Equity']
).round(2)

ratios['Debt_to_EBITDA'] = (
    master['Bank_Borrowings'] / master['EBITDA']
).round(2)

# --- Liquidity ---
ratios['Cash_to_Borrowings_%'] = (
    master['Cash'] / master['Bank_Borrowings'] * 100
).round(1)

# --- Print ---
print("=" * 60)
print("FUNDING CIRCLE — RATIO ANALYSIS")
print("=" * 60)
print(ratios.to_string())

FUNDING CIRCLE — RATIO ANALYSIS
      Revenue_Growth_%  Profit_Margin_%  ROE_%  ROA_%  Cost_to_Income_%  ECL_as_%_of_Revenue  NPL_Ratio_%  ECL_Coverage_%  Debt_to_Equity  Debt_to_EBITDA  Cash_to_Borrowings_%
2023               NaN             -7.6   -4.0   -2.7              54.9                  NaN          NaN             NaN            0.28           10.22                 318.6
2024              23.1              0.5    0.4    0.2              65.6                  5.4          8.0            91.5            0.47            6.07                 184.1
2025              27.6              9.9    8.9    3.8              69.5                  9.0          6.6            90.6            1.17            8.51                  57.0


In [ ]:
# ============================================
# DATA NOTES & DISCREPANCIES
# ============================================
# 1. Pretax Profit 2024: yfinance shows £0.8m vs £3.4m in original Excel.
#    Difference due to yfinance adjusting for discontinued operations.
#    This affects 2024 Profit Margin (shows 0.5% vs 2.1% previously).
#    Source: Funding Circle 2025 Annual Report, p.xx confirms £3.4m.
#    Not material to overall credit assessment.
#
# 2. Revenue 2025: yfinance shows £222.5m vs £204.3m in original Excel.
#    yfinance includes full group revenue. New figure more accurate.
#
# 3. ECL, NPL, Loan Book: NaN for 2023 — not separately reported
#    in 2023 filing. Genuine data gap, not a code error.
#
# 4. Interest Expense: Not available via yfinance for 2023-2025.
#    Estimated using SONIA + spread per Note 16, 2025 Annual Report.
#    See Cell 3 for methodology.

In [26]:
# ============================================
# CELL 3 — ICR & DSCR ANALYSIS
# All inputs from master — do not hardcode
# ============================================

icr_dscr = pd.DataFrame(index=years)

# --- Step 1: Calculate borrowing rate from master assumptions ---
borrowing_rate = SONIA_RATE + BORROWING_SPREAD

# --- Step 2: Estimate interest on bank borrowings ---
icr_dscr['Bank_Borrowings']       = master['Bank_Borrowings']
icr_dscr['Interest_On_Borrowings'] = (
    master['Bank_Borrowings'] * borrowing_rate
).round(2)

icr_dscr['Lease_Interest']         = master['Lease_Interest']

icr_dscr['Total_Interest_Expense'] = (
    icr_dscr['Interest_On_Borrowings'] + 
    icr_dscr['Lease_Interest']
).round(2)

# --- Step 3: ICR and DSCR ---
icr_dscr['EBIT']   = master['EBIT']
icr_dscr['EBITDA'] = master['EBITDA']

icr_dscr['ICR']  = (
    icr_dscr['EBIT'] / icr_dscr['Total_Interest_Expense']
).round(2)

icr_dscr['DSCR'] = (
    icr_dscr['EBITDA'] / icr_dscr['Total_Interest_Expense']
).round(2)

# --- Step 4: Threshold flags ---
def icr_flag(icr):
    if icr < 1.0:
        return 'RED - CONCERN'
    elif icr < 1.5:
        return 'AMBER - WARNING'
    else:
        return 'GREEN - ADEQUATE'

def dscr_flag(dscr):
    if dscr < 1.0:
        return 'RED - CONCERN'
    elif dscr < 1.5:
        return 'AMBER - WARNING'
    else:
        return 'GREEN - ADEQUATE'

icr_dscr['ICR_Flag']  = icr_dscr['ICR'].apply(icr_flag)
icr_dscr['DSCR_Flag'] = icr_dscr['DSCR'].apply(dscr_flag)

# --- Step 5: Auto memo commentary ---
latest = icr_dscr.loc[2025]
prior  = icr_dscr.loc[2024]

memo_icr = f"""
MEMO COMMENTARY — Debt Service & Interest Coverage
---------------------------------------------------
Interest coverage improved from {prior['ICR']}x in 2024 to {latest['ICR']}x
in 2025, reflecting the turnaround in operating profitability. However,
ICR remains below the 1.5x threshold considered adequate for a lender
of this profile, indicating operating earnings provide only thin
coverage of financing costs.

On an EBITDA basis, DSCR stands at {latest['DSCR']}x (2024: {prior['DSCR']}x),
offering marginally more comfort but still below the 2.0x level that
would signal strong debt serviceability.

Bank borrowings increased sharply from £{prior['Bank_Borrowings']}m to
£{latest['Bank_Borrowings']}m in 2025, funding rapid loan book expansion.
While revenue growth supports this trajectory, the pace of borrowing
growth introduces refinancing and liquidity risk if loan performance
deteriorates.

Estimated interest expense on bank borrowings: £{latest['Interest_On_Borrowings']:.1f}m (2025).
Note: Figures estimated using SONIA ({SONIA_RATE*100:.1f}%) + spread 
({BORROWING_SPREAD*100:.1f}%) per Note 16 of the 2025 Annual Report.
"""

# --- Print ---
print("=" * 70)
print("FUNDING CIRCLE — ICR & DSCR ANALYSIS")
print("=" * 70)
print(icr_dscr[['EBIT','EBITDA','Total_Interest_Expense',
                 'ICR','ICR_Flag','DSCR','DSCR_Flag']].to_string())
print(memo_icr)

FUNDING CIRCLE — ICR & DSCR ANALYSIS
      EBIT  EBITDA  Total_Interest_Expense   ICR         ICR_Flag  DSCR         DSCR_Flag
2023  -9.9     6.8                    4.52 -2.19    RED - CONCERN  1.50  GREEN - ADEQUATE
2024   3.8    16.8                    7.22  0.53    RED - CONCERN  2.33  GREEN - ADEQUATE
2025  20.4    31.4                   17.97  1.14  AMBER - WARNING  1.75  GREEN - ADEQUATE

MEMO COMMENTARY — Debt Service & Interest Coverage
---------------------------------------------------
Interest coverage improved from 0.53x in 2024 to 1.14x
in 2025, reflecting the turnaround in operating profitability. However,
ICR remains below the 1.5x threshold considered adequate for a lender
of this profile, indicating operating earnings provide only thin
coverage of financing costs.

On an EBITDA basis, DSCR stands at 1.75x (2024: 2.33x),
offering marginally more comfort but still below the 2.0x level that
would signal strong debt serviceability.

Bank borrowings increased sharply from £

In [27]:
# ============================================
# CELL 4 — INTERNAL RISK GRADE
# All inputs from master and icr_dscr
# ============================================

def assign_risk_grade(icr, dscr, profit_margin, 
                      ecl_pct_revenue, debt_to_equity):
    score = 0

    # ICR scoring (max 4)
    if icr >= 2.0:        score += 4
    elif icr >= 1.5:      score += 3
    elif icr >= 1.0:      score += 2
    elif icr >= 0:        score += 1
    else:                 score += 0

    # DSCR scoring (max 4)
    if dscr >= 2.5:       score += 4
    elif dscr >= 2.0:     score += 3
    elif dscr >= 1.5:     score += 2
    elif dscr >= 1.0:     score += 1
    else:                 score += 0

    # Profit margin scoring (max 3)
    if profit_margin >= 20:    score += 3
    elif profit_margin >= 10:  score += 2
    elif profit_margin >= 0:   score += 1
    else:                      score += 0

    # ECL as % of revenue (max 3, higher ECL = worse)
    if ecl_pct_revenue <= 3:   score += 3
    elif ecl_pct_revenue <= 6: score += 2
    elif ecl_pct_revenue <= 10: score += 1
    else:                      score += 0

    # Debt to equity (max 3, higher leverage = worse)
    if debt_to_equity <= 0.3:  score += 3
    elif debt_to_equity <= 0.6: score += 2
    elif debt_to_equity <= 1.0: score += 1
    else:                       score += 0

    # Grade mapping (max score = 17)
    if score >= 14:    grade = 'BB'
    elif score >= 11:  grade = 'B+'
    elif score >= 8:   grade = 'B'
    elif score >= 5:   grade = 'B-'
    else:              grade = 'CCC'

    return grade, score

# --- Pull inputs from master and icr_dscr ---
grade_results = pd.DataFrame(index=years)

grade_results['ICR']             = icr_dscr['ICR']
grade_results['DSCR']            = icr_dscr['DSCR']
grade_results['Profit_Margin_%'] = ratios['Profit_Margin_%']
grade_results['ECL_%_Revenue']   = ratios['ECL_as_%_of_Revenue']
grade_results['Debt_to_Equity']  = ratios['Debt_to_Equity']

# --- Apply grade function to each year ---
for year in years:
    row = grade_results.loc[year]
    
    # Handle NaN for ECL (2023)
    ecl_val = row['ECL_%_Revenue'] if not pd.isna(row['ECL_%_Revenue']) else 5.0

    grade, score = assign_risk_grade(
        icr           = row['ICR'],
        dscr          = row['DSCR'],
        profit_margin = row['Profit_Margin_%'],
        ecl_pct_revenue = ecl_val,
        debt_to_equity  = row['Debt_to_Equity']
    )
    grade_results.loc[year, 'Score'] = score
    grade_results.loc[year, 'Grade'] = grade

# --- Latest year memo commentary ---
latest_grade = grade_results.loc[2025]
prior_grade  = grade_results.loc[2024]

memo_grade = f"""
MEMO COMMENTARY — Internal Risk Grade
--------------------------------------
Funding Circle is assigned an internal risk grade of 
{latest_grade['Grade']} (score: {int(latest_grade['Score'])}/17) 
for the year ended 31 December 2025.

This reflects improving profitability trajectory (profit margin 
{latest_grade['Profit_Margin_%']}%) and adequate DSCR of {latest_grade['DSCR']}x, 
offset by thin interest coverage of {latest_grade['ICR']}x, rising ECL 
as a proportion of revenue ({latest_grade['ECL_%_Revenue']}%), and 
debt-to-equity of {latest_grade['Debt_to_Equity']}x following rapid 
balance sheet expansion in 2025.

Grade improved from {prior_grade['Grade']} in 2024, reflecting 
the operational turnaround, but remains in sub-investment grade 
territory requiring active monitoring.
"""

# --- Print ---
print("=" * 60)
print("FUNDING CIRCLE — INTERNAL RISK GRADE")
print("=" * 60)
print(grade_results.to_string())
print(memo_grade)

FUNDING CIRCLE — INTERNAL RISK GRADE
       ICR  DSCR  Profit_Margin_%  ECL_%_Revenue  Debt_to_Equity  Score Grade
2023 -2.19  1.50             -7.6            NaN            0.28    7.0    B-
2024  0.53  2.33              0.5            5.4            0.47    9.0     B
2025  1.14  1.75              9.9            9.0            1.17    6.0    B-

MEMO COMMENTARY — Internal Risk Grade
--------------------------------------
Funding Circle is assigned an internal risk grade of 
B- (score: 6/17) 
for the year ended 31 December 2025.

This reflects improving profitability trajectory (profit margin 
9.9%) and adequate DSCR of 1.75x, 
offset by thin interest coverage of 1.14x, rising ECL 
as a proportion of revenue (9.0%), and 
debt-to-equity of 1.17x following rapid 
balance sheet expansion in 2025.

Grade improved from B in 2024, reflecting 
the operational turnaround, but remains in sub-investment grade 
territory requiring active monitoring.



In [20]:
# ============================================
# CELL 5 — COVENANT SUGGESTION
# Thresholds set based on analyst judgment
# All ratio inputs from master and ratios df
# ============================================

# --- Covenant thresholds (analyst judgment) ---
ICR_COVENANT        = 1.0   # minimum ICR
DE_COVENANT         = 2.0   # maximum debt to equity
TESTING_FREQUENCY   = 'Quarterly'

# --- Current position vs covenant ---
covenants = pd.DataFrame(index=years)

covenants['ICR_Actual']           = icr_dscr['ICR']
covenants['ICR_Covenant']         = ICR_COVENANT
covenants['ICR_Headroom']         = (
    icr_dscr['ICR'] - ICR_COVENANT
).round(2)
covenants['ICR_Status'] = covenants['ICR_Headroom'].apply(
    lambda x: 'BREACH' if x < 0 else 
              'TIGHT' if x < 0.2 else 'OK'
)

covenants['DE_Actual']            = ratios['Debt_to_Equity']
covenants['DE_Covenant']          = DE_COVENANT
covenants['DE_Headroom']          = (
    DE_COVENANT - ratios['Debt_to_Equity']
).round(2)
covenants['DE_Status'] = covenants['DE_Headroom'].apply(
    lambda x: 'BREACH' if x < 0 else 
              'TIGHT' if x < 0.3 else 'OK'
)

# --- Memo commentary ---
latest_cov = covenants.loc[2025]

memo_covenant = f"""
MEMO COMMENTARY — Proposed Covenant Package
--------------------------------------------
Given the B- internal risk grade, the following financial 
covenants are recommended, tested {TESTING_FREQUENCY}:

1. Minimum Interest Coverage Ratio: {ICR_COVENANT}x
   Current position: {latest_cov['ICR_Actual']}x
   Headroom: {latest_cov['ICR_Headroom']}x
   Status: {latest_cov['ICR_Status']}

2. Maximum Debt-to-Equity Ratio: {DE_COVENANT}x
   Current position: {latest_cov['DE_Actual']}x
   Headroom: {latest_cov['DE_Headroom']}x
   Status: {latest_cov['DE_Status']}

Rationale: ICR covenant set at {ICR_COVENANT}x reflects current 
thin coverage of {latest_cov['ICR_Actual']}x, providing a buffer 
against earnings deterioration without triggering immediate breach. 
Debt-to-equity ceiling of {DE_COVENANT}x permits continued loan book 
expansion while capping leverage risk given rapid borrowing growth 
in 2025.

Breach consequence: Covenant breach triggers a review period 
of 30 days, after which the lender may require early repayment 
or renegotiation of facility terms.

Testing frequency: {TESTING_FREQUENCY} — aligned with 
Funding Circle's financial reporting cycle.
"""

# --- Print ---
print("=" * 65)
print("FUNDING CIRCLE — COVENANT ANALYSIS")
print("=" * 65)
print(covenants.to_string())
print(memo_covenant)

FUNDING CIRCLE — COVENANT ANALYSIS
      ICR_Actual  ICR_Covenant  ICR_Headroom ICR_Status  DE_Actual  DE_Covenant  DE_Headroom DE_Status
2023       -2.19           1.0         -3.19     BREACH       0.28          2.0         1.72        OK
2024        0.53           1.0         -0.47     BREACH       0.47          2.0         1.53        OK
2025        1.14           1.0          0.14      TIGHT       1.17          2.0         0.83        OK

MEMO COMMENTARY — Proposed Covenant Package
--------------------------------------------
Given the B- internal risk grade, the following financial 
covenants are recommended, tested Quarterly:

1. Minimum Interest Coverage Ratio: 1.0x
   Current position: 1.14x
   Headroom: 0.14x
   Status: TIGHT

2. Maximum Debt-to-Equity Ratio: 2.0x
   Current position: 1.17x
   Headroom: 0.83x
   Status: OK

Rationale: ICR covenant set at 1.0x reflects current 
thin coverage of 1.14x, providing a buffer 
against earnings deterioration without triggering immedi

In [21]:
# ============================================
# CELL 6 — STRESS SCENARIO ANALYSIS
# Base / Downside / Severe Downside
# All inputs from master and icr_dscr
# ============================================

# --- Base year: 2025 actuals ---
base_revenue    = master.loc[2025, 'Revenue']
base_ebit       = master.loc[2025, 'EBIT']
base_ebitda     = master.loc[2025, 'EBITDA']
base_ecl        = master.loc[2025, 'ECL']
base_borrowings = master.loc[2025, 'Bank_Borrowings']
base_interest   = icr_dscr.loc[2025, 'Total_Interest_Expense']
base_equity     = master.loc[2025, 'Equity']

# --- Scenario assumptions ---
scenarios = {
    'Base': {
        'revenue_growth':    0.00,   # no change
        'ecl_shock':         0.00,   # no change
        'sonia_shock':       0.00,   # no rate change
    },
    'Downside': {
        'revenue_growth':   -0.10,   # revenue drops 10%
        'ecl_shock':         0.30,   # ECL rises 30%
        'sonia_shock':       0.01,   # SONIA up 100bps
    },
    'Severe': {
        'revenue_growth':   -0.20,   # revenue drops 20%
        'ecl_shock':         0.50,   # ECL rises 50%
        'sonia_shock':       0.02,   # SONIA up 200bps
    }
}

# --- Calculate stressed metrics ---
results = []

for name, s in scenarios.items():

    # Stressed revenue
    stressed_revenue = base_revenue * (1 + s['revenue_growth'])

    # Stressed ECL
    stressed_ecl = base_ecl * (1 + s['ecl_shock'])

    # ECL increase hits EBIT directly
    ecl_increase = stressed_ecl - base_ecl
    stressed_ebit   = base_ebit   - ecl_increase - \
                      (base_revenue - stressed_revenue) * 0.5
    stressed_ebitda = base_ebitda - ecl_increase - \
                      (base_revenue - stressed_revenue) * 0.5

    # Stressed interest expense
    stressed_rate     = (SONIA_RATE + s['sonia_shock']) + BORROWING_SPREAD
    stressed_interest = (base_borrowings * stressed_rate) + \
                         master.loc[2025, 'Lease_Interest']

    # Stressed ratios
    stressed_icr  = round(stressed_ebit   / stressed_interest, 2)
    stressed_dscr = round(stressed_ebitda / stressed_interest, 2)
    stressed_de   = round(base_borrowings / base_equity, 2)

    # ICR status
    if stressed_icr < ICR_COVENANT:
        icr_status = 'BREACH'
    elif stressed_icr < ICR_COVENANT + 0.2:
        icr_status = 'TIGHT'
    else:
        icr_status = 'OK'

    results.append({
        'Scenario':          name,
        'Revenue_£m':        round(stressed_revenue, 1),
        'ECL_£m':            round(stressed_ecl, 1),
        'EBIT_£m':           round(stressed_ebit, 1),
        'Interest_£m':       round(stressed_interest, 1),
        'ICR':               stressed_icr,
        'ICR_Status':        icr_status,
        'DSCR':              stressed_dscr,
        'Debt_to_Equity':    stressed_de,
    })

stress_df = pd.DataFrame(results).set_index('Scenario')

# --- Memo commentary ---
base_row     = stress_df.loc['Base']
down_row     = stress_df.loc['Downside']
severe_row   = stress_df.loc['Severe']

memo_stress = f"""
MEMO COMMENTARY — Stress Scenario Analysis
-------------------------------------------
Three scenarios were modelled against 2025 actuals to assess 
downside resilience:

BASE CASE — No change from 2025 actuals
  ICR: {base_row['ICR']}x | DSCR: {base_row['DSCR']}x | Status: {base_row['ICR_Status']}

DOWNSIDE — Revenue -10%, ECL +30%, SONIA +100bps
  ICR: {down_row['ICR']}x | DSCR: {down_row['DSCR']}x | Status: {down_row['ICR_Status']}
  A moderate deterioration in credit conditions combined with 
  slower revenue growth would push ICR into breach territory,
  triggering covenant review.

SEVERE DOWNSIDE — Revenue -20%, ECL +50%, SONIA +200bps
  ICR: {severe_row['ICR']}x | DSCR: {severe_row['DSCR']}x | Status: {severe_row['ICR_Status']}
  Under severe stress, ICR falls to {severe_row['ICR']}x, well below 
  the {ICR_COVENANT}x covenant threshold. DSCR also deteriorates 
  materially, indicating debt service would be at risk.

Conclusion: The thin base case ICR of {base_row['ICR']}x leaves 
limited buffer against adverse conditions. Even a moderate 
deterioration triggers covenant breach, supporting the B- 
risk grade and Cautious Hold recommendation.
"""

# --- Print ---
print("=" * 70)
print("FUNDING CIRCLE — STRESS SCENARIO ANALYSIS")
print("=" * 70)
print(stress_df.to_string())
print(memo_stress)

FUNDING CIRCLE — STRESS SCENARIO ANALYSIS
          Revenue_£m  ECL_£m  EBIT_£m  Interest_£m   ICR ICR_Status  DSCR  Debt_to_Equity
Scenario                                                                                 
Base           222.5    18.3     20.4         18.0  1.13      TIGHT  1.75            1.17
Downside       200.2    23.8      3.8         20.6  0.18     BREACH  0.72            1.17
Severe         178.0    27.4    -11.0         23.3 -0.47     BREACH -0.00            1.17

MEMO COMMENTARY — Stress Scenario Analysis
-------------------------------------------
Three scenarios were modelled against 2025 actuals to assess 
downside resilience:

BASE CASE — No change from 2025 actuals
  ICR: 1.13x | DSCR: 1.75x | Status: TIGHT

DOWNSIDE — Revenue -10%, ECL +30%, SONIA +100bps
  ICR: 0.18x | DSCR: 0.72x | Status: BREACH
  A moderate deterioration in credit conditions combined with 
  slower revenue growth would push ICR into breach territory,
  triggering covenant review.

SEV

In [23]:
# ============================================
# CELL 7 — PROPOSED FACILITY STRUCTURE
# Terms based on credit analysis findings
# ============================================

# --- Facility terms (analyst judgment) ---
FACILITY_TYPE       = 'Revolving Credit Facility'
FACILITY_AMOUNT     = 75.0          # £m — approx 28% of current borrowings
TENOR_YEARS         = 1             # short tenor given B- grade
SONIA_MARGIN        = 0.028         # 280bps — wider than current 130bps
RENEWAL_CONDITIONS  = 'Subject to covenant compliance and annual review'
SECURITY            = 'First charge over SME loan book and receivables'
REPORTING_FREQUENCY = 'Quarterly financial statements within 45 days of period end'

# --- Calculated pricing ---
all_in_rate = SONIA_RATE + SONIA_MARGIN

# --- Current position summary for context ---
latest = {
    'icr':           icr_dscr.loc[2025, 'ICR'],
    'dscr':          icr_dscr.loc[2025, 'DSCR'],
    'grade':         grade_results.loc[2025, 'Grade'],
    'de':            ratios.loc[2025, 'Debt_to_Equity'],
    'profit_margin': ratios.loc[2025, 'Profit_Margin_%'],
}

# --- Annual interest cost to borrower ---
annual_interest = FACILITY_AMOUNT * all_in_rate

# --- Print facility structure ---
print("=" * 60)
print("FUNDING CIRCLE — PROPOSED FACILITY STRUCTURE")
print("=" * 60)
print(f"\nFacility Type:        {FACILITY_TYPE}")
print(f"Facility Amount:      £{FACILITY_AMOUNT}m")
print(f"Tenor:                {TENOR_YEARS} year")
print(f"Pricing:              SONIA + {int(SONIA_MARGIN*10000)}bps")
print(f"All-in Rate:          {all_in_rate*100:.1f}%")
print(f"Est. Annual Interest: £{annual_interest:.1f}m to lender")
print(f"Security:             {SECURITY}")
print(f"Renewal:              {RENEWAL_CONDITIONS}")
print(f"Reporting:            {REPORTING_FREQUENCY}")

print("\nFinancial Covenants (tested quarterly):")
print(f"  Minimum ICR:        {ICR_COVENANT}x  (current: {latest['icr']}x)")
print(f"  Maximum D/E:        {DE_COVENANT}x  (current: {latest['de']}x)")

# --- Memo commentary ---
memo_facility = f"""
MEMO COMMENTARY — Proposed Facility Structure
----------------------------------------------
A £{FACILITY_AMOUNT}m revolving credit facility is proposed,
representing approximately 28% of Funding Circle's current 
bank borrowings of £{base_borrowings}m. The partial commitment 
reflects the B- internal risk grade and limited headroom 
under stress scenarios.

PRICING RATIONALE:
Pricing is set at SONIA + {int(SONIA_MARGIN*10000)}bps (all-in {all_in_rate*100:.1f}%),
wider than the estimated current blended rate of SONIA + 130bps.
The additional margin compensates for:
- Thin ICR of {latest['icr']}x with limited stress buffer
- Rapid leverage expansion (D/E: {latest['de']}x)
- Rising ECL as proportion of revenue ({ratios.loc[2025, 'ECL_as_%_of_Revenue']}%)

TENOR RATIONALE:
One-year tenor with renewal subject to covenant compliance.
Short duration appropriate given B- grade and stress analysis
showing covenant breach under moderate downside conditions.
Annual renewal creates a natural review point aligned with
Funding Circle's financial reporting cycle.

SECURITY RATIONALE:
First charge over SME loan book (£{master.loc[2025, 'Loan_Book']}m)
and receivables provides direct recourse to primary income-
generating assets in event of default.

RECOMMENDATION: CAUTIOUS HOLD
Facility approved subject to quarterly covenant testing,
annual renewal review, and monitoring of ECL trajectory.
Deterioration in ICR below {ICR_COVENANT}x or D/E above 
{DE_COVENANT}x triggers immediate review.
"""

print(memo_facility)

# --- Save to Excel ---
stress_df.to_excel(
    r"C:\Users\DELL\Desktop\FundingCircle_CreditAnalysis\stress_scenarios.xlsx"
)
grade_results.to_excel(
    r"C:\Users\DELL\Desktop\FundingCircle_CreditAnalysis\risk_grade.xlsx"
)
covenants.to_excel(
    r"C:\Users\DELL\Desktop\FundingCircle_CreditAnalysis\covenants.xlsx"
)
print("All files saved successfully.")

FUNDING CIRCLE — PROPOSED FACILITY STRUCTURE

Facility Type:        Revolving Credit Facility
Facility Amount:      £75.0m
Tenor:                1 year
Pricing:              SONIA + 280bps
All-in Rate:          8.0%
Est. Annual Interest: £6.0m to lender
Security:             First charge over SME loan book and receivables
Renewal:              Subject to covenant compliance and annual review
Reporting:            Quarterly financial statements within 45 days of period end

Financial Covenants (tested quarterly):
  Minimum ICR:        1.0x  (current: 1.14x)
  Maximum D/E:        2.0x  (current: 1.17x)

MEMO COMMENTARY — Proposed Facility Structure
----------------------------------------------
A £75.0m revolving credit facility is proposed,
representing approximately 28% of Funding Circle's current 
bank borrowings of £267.3m. The partial commitment 
reflects the B- internal risk grade and limited headroom 
under stress scenarios.

PRICING RATIONALE:
Pricing is set at SONIA + 280bps (al

In [28]:
# ============================================
# BOE BENCHMARK VALIDATION
# Source: BoE Financial Stability Report 2025
# ============================================

boe_validation = """
BANK OF ENGLAND BENCHMARK VALIDATION
--------------------------------------
Source: Bank of England Financial Stability Report, 2025
Section 3.3 — UK Corporate Debt Vulnerabilities

1. ICR CONTEXT
BoE Chart 3.3 shows approximately 40-50% of UK corporates
currently have ICR below 2.5x — the threshold used to identify
vulnerable borrowers. Funding Circle's ICR of 1.14x places it
in the lower end of this vulnerable segment, consistent with
the B- internal risk grade assigned.

2. SME SECTOR STRESS
BoE notes SME arrears on non-government guaranteed debt rising
to 1.5% as of August 2025, with SMEs under more pressure than
larger corporates. Sectors most stressed: accommodation, food,
construction, retail — all key Funding Circle borrower segments.
This directly supports the rising ECL trajectory (£18.3m in 2025
vs £8.6m in 2024) observed in Funding Circle's own accounts.

3. MARKET-BASED FINANCE RISK
BoE explicitly flags that highly leveraged corporates relying on
market-based finance face refinancing challenges if credit spreads
widen. Funding Circle's £267.3m warehouse facility at SONIA+130bps
is directly exposed to this risk — consistent with our severe
downside scenario assuming SONIA +200bps.

4. INSOLVENCY CONTEXT
Monthly insolvency rate at 53 per 10,000 firms — below long-term
average of 70. This suggests the current environment is not yet
at stress levels, supporting the base case scenario assumptions.
However the BoE notes an uptick in medium-sized corporate
insolvencies which warrants monitoring.

CONCLUSION:
BoE data independently supports three key judgments in this memo:
- B- risk grade reflecting thin ICR in context of broader
  UK corporate vulnerability
- Rising ECL assumptions consistent with SME sector stress
- Severe downside scenario assumptions on funding costs
  grounded in BoE sensitivity analysis
"""

print("=" * 60)
print("BOE BENCHMARK VALIDATION")
print("=" * 60)
print(boe_validation)

BOE BENCHMARK VALIDATION

BANK OF ENGLAND BENCHMARK VALIDATION
--------------------------------------
Source: Bank of England Financial Stability Report, 2025
Section 3.3 — UK Corporate Debt Vulnerabilities

1. ICR CONTEXT
BoE Chart 3.3 shows approximately 40-50% of UK corporates
currently have ICR below 2.5x — the threshold used to identify
vulnerable borrowers. Funding Circle's ICR of 1.14x places it
in the lower end of this vulnerable segment, consistent with
the B- internal risk grade assigned.

2. SME SECTOR STRESS
BoE notes SME arrears on non-government guaranteed debt rising
to 1.5% as of August 2025, with SMEs under more pressure than
larger corporates. Sectors most stressed: accommodation, food,
construction, retail — all key Funding Circle borrower segments.
This directly supports the rising ECL trajectory (£18.3m in 2025
vs £8.6m in 2024) observed in Funding Circle's own accounts.

3. MARKET-BASED FINANCE RISK
BoE explicitly flags that highly leveraged corporates relying on
